# Práctica — Clase 9: Colab, GPU/TPU y migración a TensorFlow 2.0

Curso **Fundamentos de TensorFlow desde cero** — DIH DATAlife

En esta práctica vas a resolver dos ejercicios:

1. Medir el tiempo de entrenamiento de un modelo con y sin uso explícito de GPU (Graphics Processing Unit)
2. Reescribir un fragmento de código de estilo TensorFlow 1.x (con `Session` y `placeholder`) al estilo TensorFlow 2.x

Intenta completar cada bloque marcado con `# TODO:` antes de mirar la solución. Ejecuta este notebook en Google Colab; si activas GPU desde **Entorno de ejecución > Cambiar tipo de entorno de ejecución**, el ejercicio 1 será más ilustrativo, pero también funciona sin GPU (simplemente verás tiempos similares en ambos bloques).

In [ ]:
import tensorflow as tf
import time

print("Versión de TensorFlow:", tf.__version__)
print("GPU disponible:", bool(tf.config.list_physical_devices('GPU')))

## Ejercicio 1 — Comparar el tiempo de entrenamiento con y sin GPU

**Enunciado**: vas a entrenar dos veces el mismo modelo denso sobre el dataset MNIST (dígitos escritos a mano): una vez forzando la ejecución en CPU (Central Processing Unit) y otra dejando que TensorFlow use la GPU si está disponible. Mide el tiempo de cada entrenamiento con `time.time()` y compáralos.

**Pistas**:

- Usa el gestor de contexto `with tf.device('/CPU:0'):` para forzar la CPU
- Guarda el instante inicial antes de llamar a `fit()` y el instante final justo después, y calcula la diferencia
- Si no hay GPU activada en tu entorno, comprueba primero con `tf.config.list_physical_devices('GPU')` y adapta el mensaje que imprimes en lugar de forzar un dispositivo que no existe
- Usa `epochs=1` y un `batch_size` moderado (por ejemplo 128) para que el ejercicio no tarde demasiado en cualquiera de los dos casos

In [ ]:
# Datos y modelo ya preparados: no hace falta que los modifiques
(x_train, y_train), _ = tf.keras.datasets.mnist.load_data()
x_train = x_train.reshape(-1, 784).astype("float32") / 255.0

def crear_modelo():
    return tf.keras.Sequential([
        tf.keras.layers.Dense(256, activation='relu', input_shape=(784,)),
        tf.keras.layers.Dense(256, activation='relu'),
        tf.keras.layers.Dense(10, activation='softmax')
    ])

def entrenar_y_medir(dispositivo, epocas=1, batch_size=128):
    """Entrena crear_modelo() en el dispositivo indicado y devuelve el tiempo en segundos."""
    # TODO: crea el modelo con crear_modelo() dentro de un bloque "with tf.device(dispositivo):"
    # TODO: compílalo con optimizer='adam' y loss='sparse_categorical_crossentropy'
    # TODO: guarda el tiempo antes de entrenar (time.time())
    # TODO: llama a fit(x_train, y_train, epochs=epocas, batch_size=batch_size, verbose=0)
    # TODO: calcula y devuelve el tiempo transcurrido (tiempo final - tiempo inicial)
    pass

# TODO: llama a entrenar_y_medir('/CPU:0') y guarda el resultado en tiempo_cpu
# TODO: comprueba si hay GPU disponible; si la hay, llama a entrenar_y_medir('/GPU:0')
#       y guarda el resultado en tiempo_gpu; si no la hay, imprime un aviso
# TODO: imprime ambos tiempos y, si mediste los dos, cuántas veces más rápida fue la GPU

<details>
<summary>💡 Ver solución</summary>

```python
def entrenar_y_medir(dispositivo, epocas=1, batch_size=128):
    """Entrena crear_modelo() en el dispositivo indicado y devuelve el tiempo en segundos."""
    with tf.device(dispositivo):
        modelo = crear_modelo()
        modelo.compile(optimizer='adam', loss='sparse_categorical_crossentropy')
        inicio = time.time()
        modelo.fit(x_train, y_train, epochs=epocas, batch_size=batch_size, verbose=0)
        fin = time.time()
    return fin - inicio

# Entrenamiento forzado en CPU
tiempo_cpu = entrenar_y_medir('/CPU:0')
print(f"Tiempo en CPU: {tiempo_cpu:.2f} segundos")

# Entrenamiento en GPU, solo si el entorno de ejecución tiene una activada
if tf.config.list_physical_devices('GPU'):
    tiempo_gpu = entrenar_y_medir('/GPU:0')
    print(f"Tiempo en GPU: {tiempo_gpu:.2f} segundos")
    print(f"La GPU fue {tiempo_cpu / tiempo_gpu:.1f} veces más rápida que la CPU")
else:
    print("No hay GPU activada en este entorno: activa GPU en Entorno de ejecución "
          "> Cambiar tipo de entorno de ejecución para comparar tiempos reales")
```
</details>

## Ejercicio 2 — De TensorFlow 1.x a TensorFlow 2.x

**Enunciado**: el siguiente fragmento está escrito al estilo TensorFlow 1.x: usa `tf.compat.v1.placeholder()` para reservar la entrada, construye el grafo de una regresión lineal muy simple (`y = w * x + b`) y necesita una `Session` para calcular el resultado. Reescríbelo al estilo TensorFlow 2.x: sin placeholders, sin sesión, con ejecución eager.

Código original (TensorFlow 1.x):

```python
import tensorflow.compat.v1 as tf1
tf1.disable_eager_execution()

x = tf1.placeholder(tf1.float32, shape=(None,))
w = tf1.Variable(2.0, dtype=tf1.float32)
b = tf1.Variable(1.0, dtype=tf1.float32)
y = w * x + b

init = tf1.global_variables_initializer()
with tf1.Session() as sess:
    sess.run(init)
    resultado = sess.run(y, feed_dict={x: [1.0, 2.0, 3.0]})
    print(resultado)
```

**Pistas**:

- En TensorFlow 2.x los datos de entrada se pasan directamente como un `tf.constant(...)` o incluso una lista de Python: no hace falta ningún `placeholder`
- Una `tf.Variable` está lista para usarse en el momento en que la creas, sin `global_variables_initializer()`
- El resultado se obtiene leyendo `.numpy()` sobre el tensor calculado, sin `Session.run()`

In [ ]:
import tensorflow as tf

# TODO: define x como un tf.constant con los valores [1.0, 2.0, 3.0]
# TODO: define w y b como tf.Variable, con los mismos valores iniciales que el original (2.0 y 1.0)
# TODO: calcula y = w * x + b directamente, sin placeholder ni Session
# TODO: imprime el resultado usando .numpy()

<details>
<summary>💡 Ver solución</summary>

```python
import tensorflow as tf

# Los datos de entrada se pasan directamente, sin placeholder
x = tf.constant([1.0, 2.0, 3.0], dtype=tf.float32)

# Las variables están listas para usarse nada más crearlas, sin inicializador aparte
w = tf.Variable(2.0, dtype=tf.float32)
b = tf.Variable(1.0, dtype=tf.float32)

# La operación se calcula de inmediato (ejecución eager), sin Session ni sess.run()
y = w * x + b

# El resultado ya es un tensor calculado: .numpy() lo convierte a un array de NumPy
print(y.numpy())  # [3. 5. 7.]
```
</details>

## Para terminar

Si te ha sobrado tiempo, prueba a repetir el ejercicio 1 subiendo `epochs` a 3 o 5 y observa si la diferencia entre CPU y GPU se hace más o menos notable a medida que el entrenamiento es más largo. Y revisa el capítulo 11 del manual si quieres repasar por qué una TPU (Tensor Processing Unit) puede ser todavía más rápida que una GPU para este tipo de tareas.